In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Note: This notebook is for reference and educational purposes only. Not intended for production use.

# Questions? mateuswagner at google.com

<small>

# AI Translation with Quality Scoring
## What This Script Does

This notebook uses **Gemini AI** (via Vertex AI) to translate home furnishings e-commerce product data and automatically score translation quality.

---

### Steps

1. **Load input data** — Reads an Excel file with product text rows, each containing a source locale, target locale, and original product value (titles, descriptions, dimensions).

2. **Translate with Gemini** — Calls **Gemini 3 Flash** to translate each row. The model preserves measurements (e.g. `8' x 6"`), applies locale-specific formatting, and uses e-commerce terminology. Token counts and latency are captured per call (retries up to 3x on failure).

3. **Judge translation quality** — A second model (**Gemini 3.1 Pro**) scores each Gemini translation from **0–5**, comparing it against the original and a Google Translate API reference. A justification is also returned. Each row's full result is saved as a JSON file.

4. **Export results** — All JSON files are compiled into a single Excel output with translations, token usage, judge scores, and justifications.

---

### Input / Output

| | File | Description |
|---|---|---|
| **Input** | `Sampling Data_Translation_March 2026.xlsx` | Columns: `sourceLocale`, `targetLocale`, `originalValue`, `translatedValue` (Translate API reference) |
| **Output** | `output-<timestamp>/Translation_Results.xlsx` | Gemini translation, token counts, judge score (0–5), and justification per row |
| **Output** | `output-<timestamp>/raw/row_XXXX.json` | Raw request/response/judge data per row |

### This notebook is for reference and educational purposes only. Not intended for production use.
</small>

In [ ]:
# Use a fresh .venv and update libraries to the latest.
!pip install -U google-genai pandas openpyxl google-cloud-aiplatform


In [ ]:
"""Imports."""

import json
import os
import time
from datetime import datetime
from pathlib import Path

from google import genai
from google.api_core import exceptions
from google.genai import types
import pandas as pd

In [ ]:
"""Constants."""
#How to migrate to the latest version
#https://docs.cloud.google.com/vertex-ai/generative-ai/docs/migrate#migration-steps
MODEL_NAME = "gemini-3-flash-preview"
JUDGE_MODEL_ID = "gemini-3.1-pro-preview"

INPUT_FILE = "Sampling Data_Translation_March 2026.xlsx"
OUTPUT_FILE_NAME = "Translation_Results.xlsx"

# Prompt
PROMPT_TEMPLATE = (
    "Translate the following home furnishings product text from {source_locale} "
    "to {target_locale}. This is e-commerce product data (titles, descriptions, "
    "dimensions) for items such as furniture, rugs, lighting, and décor.\n\n"
    "Guidelines:\n"
    "- Preserve product measurements and dimensions "
    "(e.g., 8', 6\" x 9\") as-is.\n"
    "- Interpret shape and size terms in a product context "
    "(e.g., \"Round 8'\" = circular, 8-foot diameter).\n"
    "- Apply locale-specific formatting, spelling, and conventions for "
    "{target_locale}.\n\n"
    "Text to translate:\n{original_value}"
)

# Judge prompt
JUDGE_PROMPT_TEMPLATE = (
    "You are a translation quality judge for home furnishings "
    "e-commerce content.\n\n"
    "Evaluate the quality of the following Gemini translation by comparing it "
    "to the original text. A reference translation from the Translate API is "
    "provided for additional context.\n\n"
    "Source language: {source_locale}\n"
    "Target language: {target_locale}\n\n"
    "Original text:\n{original_value}\n\n"
    "Translate API translation (reference):\n{translate_api_value}\n\n"
    "Gemini translation (to evaluate):\n{gemini_value}\n\n"
    "Rate the Gemini translation on a scale of 0 to 5:\n"
    "  0 = Completely wrong or nonsensical\n"
    "  1 = Major errors that change meaning\n"
    "  2 = Multiple significant errors\n"
    "  3 = Acceptable with minor errors\n"
    "  4 = Good with very minor issues\n"
    "  5 = Excellent, accurate and natural\n\n"
    "Provide the score and a brief justification."
)
JUDGE_RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "score": {"type": "INTEGER"},
        "justification": {"type": "STRING"},
    },
    "required": ["score", "justification"],
}

# Generation config
# Gemini 3 strongly recommends keeping temperature at its default value of 1.0.
TEMPERATURE = 1.0
TOP_P = 0.95
SEED = 7
MAX_OUTPUT_TOKENS = 65535
RESPONSE_MIME_TYPE = "application/json"
SYSTEM_INSTRUCTION = (
    "You are an expert localizer for a home furnishings and décor e-commerce "
    "company. You translate product catalog data (titles, descriptions, "
    "attributes) accurately, preserving product specifications and industry "
    "terminology."
)
THINKING_LEVEL = "MINIMAL"
SAFETY_THRESHOLD = "OFF"
SAFETY_CATEGORIES = [
    "HARM_CATEGORY_HATE_SPEECH",
    "HARM_CATEGORY_DANGEROUS_CONTENT",
    "HARM_CATEGORY_SEXUALLY_EXPLICIT",
    "HARM_CATEGORY_HARASSMENT",
]
RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "translatedValue": {"type": "STRING"},
    },
    "required": ["translatedValue"],
}

# Processing config
MAX_ROWS_FOR_TEST = None  # Set to None to process all rows.
MAX_RETRIES = 3
RETRY_WAIT_SECS = 1

In [ ]:
"""Helper functions for translation."""


def create_client() -> genai.Client:
    """Creates and returns a Vertex AI Gemini client."""
    return genai.Client(
        vertexai=True,
        #api_key=os.environ.get(API_KEY_ENV_VAR),
    )


def build_prompt(
    source_locale: str,
    target_locale: str,
    original_value: str,
) -> str:
    """Builds a translation prompt for a single row.

    Args:
        source_locale: BCP-47 locale of the source text.
        target_locale: BCP-47 locale for the translation.
        original_value: The text to translate.

    Returns:
        A formatted prompt string.
    """
    return PROMPT_TEMPLATE.format(
        source_locale=source_locale,
        target_locale=target_locale,
        original_value=original_value,
    )


def _make_safety_settings() -> list[types.SafetySetting]:
    """Returns safety settings with all categories disabled."""
    return [
        types.SafetySetting(category=cat, threshold=SAFETY_THRESHOLD)
        for cat in SAFETY_CATEGORIES
    ]


def get_generation_config() -> types.GenerateContentConfig:
    """Returns the generation config for translation."""
    return types.GenerateContentConfig(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        seed=SEED,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        safety_settings=_make_safety_settings(),
        response_mime_type=RESPONSE_MIME_TYPE,
        response_schema=RESPONSE_SCHEMA,
        system_instruction=[
            types.Part.from_text(text=SYSTEM_INSTRUCTION)
        ],
        thinking_config=types.ThinkingConfig(
            thinking_level=THINKING_LEVEL
        ),
    )


def get_judge_config() -> types.GenerateContentConfig:
    """Returns the generation config for the judge model."""
    return types.GenerateContentConfig(
        temperature=0.0,
        top_p=TOP_P,
        seed=SEED,
        max_output_tokens=1024,
        safety_settings=_make_safety_settings(),
        response_mime_type=RESPONSE_MIME_TYPE,
        response_schema=JUDGE_RESPONSE_SCHEMA,
    )


def translate_row(
    client: genai.Client,
    source_locale: str,
    target_locale: str,
    original_value: str,
    debug: bool = False,
) -> dict:
    """Sends a single row to the LLM for translation.

    Args:
        client: The Gemini API client.
        source_locale: BCP-47 locale of the source text.
        target_locale: BCP-47 locale for the translation.
        original_value: The text to translate.
        debug: If True, prints usage metadata.

    Returns:
        A dict with request and response data including
        the translated text and token counts.
    """
    prompt = build_prompt(
        source_locale, target_locale, original_value
    )
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        ),
    ]

    start_time = time.time()
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config=get_generation_config(),
    )
    elapsed_ms = (time.time() - start_time) * 1000

    usage = response.usage_metadata

    if debug:
        print(f"  [DEBUG] usage_metadata: {usage}")

    input_tokens = getattr(usage, "prompt_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    thinking_tokens = getattr(usage, "thoughts_token_count", 0) or 0
    total_tokens = input_tokens + output_tokens + thinking_tokens

    result = json.loads(response.text)

    return {
        "request": {
            "sourceLocale": source_locale,
            "targetLocale": target_locale,
            "originalValue": original_value,
            "prompt": prompt,
        },
        "response": {
            "rawResponse": response.text,
            "translatedValue": result.get("translatedValue", ""),
            "inputTokens": input_tokens,
            "outputTokens": output_tokens,
            "thinkingTokens": thinking_tokens,
            "totalTokens": total_tokens,
            "durationMs": round(elapsed_ms),
        },
    }


def judge_translation(
    client: genai.Client,
    source_locale: str,
    target_locale: str,
    original_value: str,
    translate_api_value: str,
    gemini_value: str,
) -> dict:
    """Asks the judge model to score a Gemini translation.

    Args:
        client: The Gemini API client.
        source_locale: BCP-47 locale of the source text.
        target_locale: BCP-47 locale for the translation.
        original_value: The original text.
        translate_api_value: Reference translation from the
            Translate API.
        gemini_value: The Gemini translation to evaluate.

    Returns:
        A dict with score, justification, and token usage.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        source_locale=source_locale,
        target_locale=target_locale,
        original_value=original_value,
        translate_api_value=translate_api_value,
        gemini_value=gemini_value,
    )
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)],
        ),
    ]

    start_time = time.time()
    response = client.models.generate_content(
        model=JUDGE_MODEL_ID,
        contents=contents,
        config=get_judge_config(),
    )
    elapsed_ms = (time.time() - start_time) * 1000

    usage = response.usage_metadata
    input_tokens = getattr(usage, "prompt_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    total_tokens = input_tokens + output_tokens

    result = json.loads(response.text)

    return {
        "score": result.get("score", -1),
        "justification": result.get("justification", ""),
        "inputTokens": input_tokens,
        "outputTokens": output_tokens,
        "totalTokens": total_tokens,
        "durationMs": round(elapsed_ms),
    }

In [ ]:
"""Load translation data from Excel."""

df = pd.read_excel(INPUT_FILE)

print(f"Loaded {len(df)} rows from {INPUT_FILE}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
"""Process translations, judge quality, and save raw JSON files."""

# Create output folder.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(f"output-{timestamp}")
raw_dir = output_dir / "raw"
raw_dir.mkdir(parents=True)
print(f"Output folder: {output_dir}/")

client = create_client()

rows_to_process = min(MAX_ROWS_FOR_TEST or len(df), len(df))

for idx in range(rows_to_process):
    row = df.iloc[idx]
    print(
        f"Processing row {idx + 1}/{rows_to_process}: "
        f"{row['sourceLocale']} -> {row['targetLocale']} | "
        f"{row['originalValue'][:50]}..."
    )

    # Translate with retry.
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = translate_row(
                client,
                row["sourceLocale"],
                row["targetLocale"],
                row["originalValue"],
                debug=(idx == 0 and attempt == 1),
            )
            break
        except (
            exceptions.ClientError,
            exceptions.ServerError,
        ) as e:
            if attempt < MAX_RETRIES:
                print(
                    f"  [RETRY {attempt}/{MAX_RETRIES}] "
                    f"{e}. Retrying in "
                    f"{RETRY_WAIT_SECS}s..."
                )
                time.sleep(RETRY_WAIT_SECS)
            else:
                print(
                    f"  [FAILED] Row {idx + 1} after "
                    f"{MAX_RETRIES} attempts: {e}"
                )
                raise

    resp = result["response"]
    print(
        f"  -> {resp['translatedValue']} | "
        f"in:{resp['inputTokens']} "
        f"out:{resp['outputTokens']} "
        f"think:{resp['thinkingTokens']} "
        f"total:{resp['totalTokens']} | "
        f"{resp['durationMs']}ms"
    )

    # Judge the Gemini translation with retry.
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            judge_result = judge_translation(
                client,
                source_locale=row["sourceLocale"],
                target_locale=row["targetLocale"],
                original_value=row["originalValue"],
                translate_api_value=row["translatedValue"],
                gemini_value=resp["translatedValue"],
            )
            break
        except (
            exceptions.ClientError,
            exceptions.ServerError,
        ) as e:
            if attempt < MAX_RETRIES:
                print(
                    f"  [JUDGE RETRY "
                    f"{attempt}/{MAX_RETRIES}] {e}. "
                    f"Retrying in {RETRY_WAIT_SECS}s..."
                )
                time.sleep(RETRY_WAIT_SECS)
            else:
                print(
                    f"  [JUDGE FAILED] Row {idx + 1} "
                    f"after {MAX_RETRIES} attempts: {e}"
                )
                raise

    result["judge"] = judge_result
    print(
        f"  [JUDGE] score: {judge_result['score']}/5 | "
        f"{judge_result['justification'][:80]}... | "
        f"{judge_result['durationMs']}ms"
    )

    # Save raw request + response + judge as JSON.
    raw_file = raw_dir / f"row_{idx + 1:04d}.json"
    with open(raw_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

print(
    f"\nCompleted. {rows_to_process} rows saved to "
    f"{raw_dir}/"
)

In [ ]:
"""Build final results Excel from raw JSON files."""

raw_files = sorted(raw_dir.glob("row_*.json"))
print(f"Reading {len(raw_files)} raw files from {raw_dir}/...")

rows = []
for raw_file in raw_files:
    with open(raw_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    req = data["request"]
    resp = data["response"]
    judge = data.get("judge", {})
    rows.append({
        "sourceLocale": req["sourceLocale"],
        "targetLocale": req["targetLocale"],
        "originalValue": req["originalValue"],
        "translatedValue": resp["translatedValue"],
        "inputTokens": resp["inputTokens"],
        "outputTokens": resp["outputTokens"],
        "thinkingTokens": resp["thinkingTokens"],
        "totalTokens": resp["totalTokens"],
        "durationMs": resp["durationMs"],
        "judgeScore": judge.get("score", ""),
        "judgeJustification": judge.get(
            "justification", ""
        ),
        "judgeInputTokens": judge.get("inputTokens", ""),
        "judgeOutputTokens": judge.get(
            "outputTokens", ""
        ),
        "judgeDurationMs": judge.get("durationMs", ""),
    })

results_df = pd.DataFrame(rows)
output_file = output_dir / OUTPUT_FILE_NAME
results_df.to_excel(output_file, index=False)

avg_score = (
    results_df["judgeScore"]
    .replace("", pd.NA)
    .dropna()
    .astype(int)
    .mean()
)
print(
    f"Results saved to {output_file} "
    f"({len(results_df)} rows)"
)
print(f"Average judge score: {avg_score:.2f}/5")
results_df.head(10)